In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

# ============================================================
# Unit Root Test (ADF)
# - raw level data: tvpvar_raw_level_merged.csv
# - transformed/scaled data: tvpvar_input_scaled.csv
# - local VS Code / notebook 기준
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")

RAW_FILE = BASE_DIR / "tvpvar_raw_level_merged.csv"
TRANS_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

OUT_RAW = BASE_DIR / "adf_raw.csv"
OUT_TRANS = BASE_DIR / "adf_transformed.csv"
OUT_SUMMARY = BASE_DIR / "adf_summary.txt"

DATE_CANDIDATES = ["Date", "date", "DATE"]

# -----------------------------
# Helpers
# -----------------------------
def find_date_col(df: pd.DataFrame):
    for c in DATE_CANDIDATES:
        if c in df.columns:
            return c
    return None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

    df = pd.read_csv(path)

    date_col = find_date_col(df)
    if date_col is not None:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df = df.sort_values(date_col).reset_index(drop=True)

    return df


def get_numeric_columns(df: pd.DataFrame):
    num_cols = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            num_cols.append(col)
    return num_cols


def run_adf(series: pd.Series, var_name: str, regression: str = "c", autolag: str = "AIC"):
    """
    regression:
        'c'  : constant only
        'ct' : constant + trend
        'ctt': constant + trend + quadratic trend
        'n'  : no constant, no trend
    """
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

    result = {
        "variable": var_name,
        "n_obs": int(len(s)),
        "regression": regression,
        "autolag": autolag,
        "adf_stat": np.nan,
        "p_value": np.nan,
        "used_lag": np.nan,
        "n_used_for_regression": np.nan,
        "crit_1%": np.nan,
        "crit_5%": np.nan,
        "crit_10%": np.nan,
        "is_stationary_5pct": np.nan,
        "status": ""
    }

    if len(s) < 20:
        result["status"] = "too_few_observations"
        return result

    if s.nunique() <= 1:
        result["status"] = "constant_or_single_value"
        return result

    try:
        adf_res = adfuller(s, regression=regression, autolag=autolag)
        result["adf_stat"] = adf_res[0]
        result["p_value"] = adf_res[1]
        result["used_lag"] = adf_res[2]
        result["n_used_for_regression"] = adf_res[3]
        result["crit_1%"] = adf_res[4].get("1%", np.nan)
        result["crit_5%"] = adf_res[4].get("5%", np.nan)
        result["crit_10%"] = adf_res[4].get("10%", np.nan)
        result["is_stationary_5pct"] = bool(adf_res[1] < 0.05)
        result["status"] = "ok"
    except Exception as e:
        result["status"] = f"error: {str(e)}"

    return result


def adf_table(df: pd.DataFrame, dataset_name: str, regression: str = "c", autolag: str = "AIC"):
    num_cols = get_numeric_columns(df)
    rows = []

    for col in num_cols:
        res = run_adf(df[col], col, regression=regression, autolag=autolag)
        res["dataset"] = dataset_name
        rows.append(res)

    out = pd.DataFrame(rows)

    if not out.empty:
        cols = [
            "dataset", "variable", "n_obs", "regression", "autolag",
            "adf_stat", "p_value", "used_lag", "n_used_for_regression",
            "crit_1%", "crit_5%", "crit_10%", "is_stationary_5pct", "status"
        ]
        out = out[cols]

    return out


def format_summary(df_raw_res: pd.DataFrame, df_trans_res: pd.DataFrame) -> str:
    lines = []

    lines.append("============================================================")
    lines.append("ADF Unit Root Test Summary")
    lines.append("============================================================")
    lines.append("")

    def add_block(title: str, df_res: pd.DataFrame):
        lines.append(title)
        lines.append("-" * len(title))

        if df_res.empty:
            lines.append("No results.")
            lines.append("")
            return

        ok = df_res[df_res["status"] == "ok"].copy()
        fail = df_res[df_res["status"] != "ok"].copy()

        if not ok.empty:
            lines.append("Variables tested:")
            for _, row in ok.iterrows():
                lines.append(
                    f"  - {row['variable']}: "
                    f"ADF={row['adf_stat']:.6f}, "
                    f"p={row['p_value']:.6f}, "
                    f"lag={int(row['used_lag']) if pd.notna(row['used_lag']) else 'NA'}, "
                    f"stationary@5%={row['is_stationary_5pct']}"
                )
        else:
            lines.append("No successful ADF results.")

        if not fail.empty:
            lines.append("")
            lines.append("Failed / skipped variables:")
            for _, row in fail.iterrows():
                lines.append(f"  - {row['variable']}: {row['status']}")

        lines.append("")

    add_block("1) Raw level data", df_raw_res)
    add_block("2) Transformed data", df_trans_res)

    if not df_trans_res.empty:
        ok_trans = df_trans_res[df_trans_res["status"] == "ok"].copy()
        stationary_cols = ok_trans.loc[ok_trans["is_stationary_5pct"] == True, "variable"].tolist()
        non_stationary_cols = ok_trans.loc[ok_trans["is_stationary_5pct"] == False, "variable"].tolist()

        lines.append("3) Final check for transformed data")
        lines.append("----------------------------------")
        lines.append(f"Stationary @ 5%: {stationary_cols if stationary_cols else 'None'}")
        lines.append(f"Not stationary @ 5%: {non_stationary_cols if non_stationary_cols else 'None'}")
        lines.append("")

    return "\n".join(lines)


# -----------------------------
# Main
# -----------------------------
def main():
    print("[1/4] Loading data...")
    df_raw = load_csv(RAW_FILE)
    df_trans = load_csv(TRANS_FILE)

    print("Raw shape        :", df_raw.shape)
    print("Transformed shape:", df_trans.shape)

    print("\n[2/4] Running ADF on raw level data...")
    raw_res = adf_table(df_raw, dataset_name="raw_level", regression="c", autolag="AIC")

    print("[3/4] Running ADF on transformed data...")
    trans_res = adf_table(df_trans, dataset_name="transformed", regression="c", autolag="AIC")

    print("\n[4/4] Saving results...")
    raw_res.to_csv(OUT_RAW, index=False, encoding="utf-8-sig")
    trans_res.to_csv(OUT_TRANS, index=False, encoding="utf-8-sig")

    summary_text = format_summary(raw_res, trans_res)
    with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
        f.write(summary_text)

    print("\nSaved:")
    print(f" - {OUT_RAW}")
    print(f" - {OUT_TRANS}")
    print(f" - {OUT_SUMMARY}")

    print("\nPreview: transformed data ADF")
    if not trans_res.empty:
        preview_cols = ["variable", "adf_stat", "p_value", "used_lag", "is_stationary_5pct", "status"]
        print(trans_res[preview_cols].to_string(index=False))
    else:
        print("No transformed ADF results.")

    print("\nDone.")


if __name__ == "__main__":
    main()

[1/4] Loading data...
Raw shape        : (1344, 8)
Transformed shape: (1036, 8)

[2/4] Running ADF on raw level data...
[3/4] Running ADF on transformed data...

[4/4] Saving results...

Saved:
 - adf_raw.csv
 - adf_transformed.csv
 - adf_summary.txt

Preview: transformed data ADF
   variable   adf_stat      p_value  used_lag  is_stationary_5pct status
dlog_SOLVPN -30.360984 0.000000e+00         0                True     ok
dlog_COPPER -33.106097 0.000000e+00         0                True     ok
  dlog_GOLD -33.586020 0.000000e+00         0                True     ok
dlog_SILVER -34.864268 0.000000e+00         0                True     ok
   dlog_DXY -24.851222 0.000000e+00         1                True     ok
   d_UST10Y -32.042979 0.000000e+00         0                True     ok
      d_VIX -17.610739 3.880748e-30         4                True     ok

Done.
